# illust-asset-forge

上から順に実行する。

## 全体の流れ

1. **準備**（セル 1・2）: リポジトリとモデルを用意する
2. **顔**（セル 3）: 顔を作って 1 枚選ぶ → **Claude に渡す**（下の「次にやること」）
3. **表情**（セル 4）: Claude が `inpaint_box` を書いて push した後に、選んだ顔の表情を作る → **Claude に渡す**
4. **全身**（セル 5）: 全身絵を作る → **Claude に渡す**

Claude に渡すと、Claude が書き出して（顔は 256×256、全身は元の大きさ）リポジトリに入れる。

## 1. 準備（リポジトリ・ライブラリ）

Drive をつなぎ、リポジトリを取り、ライブラリを入れる。Claude が設定を直して push した後は、**このセルだけ**を実行し直せば取り込める（モデルは読み込み直さない）。

In [ ]:
import os
import subprocess
import sys
from google.colab import drive

drive.mount("/content/drive")

REPO = "/content/illust-asset-forge"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/akabee0161/illust-asset-forge.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)

!pip install -q -r {REPO}/requirements-colab.txt
!pip install -q -e {REPO} --no-deps

os.chdir(REPO)
# pip install -e は、動いているカーネルの import には効かない。src を直接足す
if os.path.join(REPO, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO, "src"))
# 実行し直したときに古いコードのままにならないよう、読み込み済みの artgen を捨てる
for name in [m for m in sys.modules if m == "artgen" or m.startswith("artgen.")]:
    del sys.modules[name]
from artgen import forge as F

## 2. モデルの読み込み

`forge` が無いときだけ読み込む（2 回目以降は何もしない）。メモリ不足で止まったときだけ、`F.load(".", low_memory=True)` にする（遅くなる）。

**終わったら** → 次の 3 の顔のセルへ。

In [ ]:
if "forge" not in globals():
    forge = F.load(".")

## 3. 顔（通常）

1 枚できるたびに表示される（1 枚 20 秒ほど）。調整中は `n=2` など少なくし、気に入らなければ ■ で止めて、パラメータを変えて作り直す（止めても、できた分は Drive に残る）。一気に作るときは `n=16` などにして、後で Drive から見る。

In [ ]:
paths = forge.generate("ally05", "face", n=8, on_image=F.show_one)

### 次にやること（顔を採用したら）

1. 画像を見て、採用する顔を **1 枚**選ぶ。目印は、画像の下に出るファイル名（`ally05_face_<シード>.png`）
2. 調整したいときは、下の「試すときの書き方」で `extra=`・`remove=` を足して、上のセルを実行し直す
3. Google Drive の「マイドライブ」→ `illust-asset-forge` → `runs` → （日時のフォルダ）から、選んだ PNG をスマホにダウンロードする
4. **zip にして**、Claude とのチャットに添付する（zip にすると、PNG に埋め込まれた生成の条件が消えにくい）。どの画像を選んだかも、ファイル名かシードで伝える
5. Claude が `inpaint_box`（目と口の範囲）を書いて push するのを待つ。「push した」と言われたら、**1 のセルを実行し直して**設定を取り込み、次の 4 の表情へ進む

## 4. 表情

採用した顔を `source` に渡す（Drive 上のパス）。`inpaint_box` が要るので、上の「次にやること」の 5 まで済んでから実行する。表情名は `character.yaml` の `expressions` のキー（`hurt`・`shout` など）。

`source` の `<run_id>` と `<採用した顔>` は、Drive の実際の名前に書き換える。

In [ ]:
paths = forge.inpaint("ally05", "hurt", source="/content/drive/MyDrive/illust-asset-forge/runs/<run_id>/<採用した顔>.png", n=4, on_image=F.show_one)

### 次にやること（表情を採用したら）

1. 表情ごとに 1 枚選ぶ。ほかの表情（`shout` など）も作るときは、上のセルの `"hurt"` を書き換えて実行する
2. 顔と同じく、選んだ PNG を Drive からダウンロードして zip にし、Claude に添付する。**どの表情の画像か**（`hurt` など）も伝える
3. 次の 5 の全身へ進む（表情の受け渡しを待たなくてよい）

## 5. 全身（参考資料）

顔と同じように、1 枚できるたびに表示される。

In [ ]:
paths = forge.generate("ally05", "fullbody", n=4, on_image=F.show_one)

### 次にやること（全身を採用したら）

1. 全身絵を 1 枚選ぶ。見るのは、服の種類・色・持ち物と、髪の色・髪型が顔と合っているか
2. 顔と同じく、Drive からダウンロードして zip にし、Claude に添付する
3. これで 1 体分は終わり。Claude が書き出して、リポジトリに入れる

## 試すときの書き方

- 足すタグ・消すタグ: `forge.generate("ally05", "face", extra="scar on cheek", remove="stubble")`（ネガティブは `extra_negative=`・`remove_negative=`）。`remove` は設定のタグにだけ効く。`extra` に設定と同じタグを重み付きで書くと、設定のタグが置き換わる（例: `extra="(stubble:1.3)"`）
- シードを固定: `seeds=[123, 456]`（渡すと `n` は無視される）
- guidance・steps（inpaint は `strength` も）: `guidance=6`、`steps=30`
- 画像を並べて見る: `F.show(paths)`
- **YAML は Colab で書き換えない**（次の pull でぶつかる）。採用した PNG を zip にして Claude に渡す。上書きの内容は PNG の条件（`overrides`）に残るので、Claude が設定に戻す。